A DataFrame is a Dataset organized into named columns. It is conceptually equivalent to a table in a relational database or a data frame in R/Python, but with richer optimizations under the hood. DataFrames can be constructed from a wide array of sources such as: structured data files, tables in Hive, external databases, or existing RDDs.
A DataFrame is an immutable distributed collection of data, only available in the current Spark session.

In [1]:
from pyspark.sql import SparkSession

spark =  SparkSession.builder \
                    .master("spark://spark-master:7077") \
                    .appName("DataFrames") \
                    .config("spark.executor.memory", "1g") \
                    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 09:45:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
employees = [
  {"name": "John D.", "age": 30, "department": "HR"},
  {"name": "Alice G.", "age": 25, "department": "Finance"},
  {"name": "Bob T.", "age": 35, "department": "IT"},
  {"name": "Eve A.", "age": 28, "department": "Marketing"}
]
df = spark.createDataFrame(employees)

In [6]:
df.printSchema()

root
 |-- age: long (nullable = true)
 |-- department: string (nullable = true)
 |-- name: string (nullable = true)



In [10]:
df.describe().show()

+-------+-----------------+----------+--------+
|summary|              age|department|    name|
+-------+-----------------+----------+--------+
|  count|                4|         4|       4|
|   mean|             29.5|      NULL|    NULL|
| stddev|4.203173404306164|      NULL|    NULL|
|    min|               25|   Finance|Alice G.|
|    max|               35| Marketing| John D.|
+-------+-----------------+----------+--------+



In [4]:
df1=df.select("name","age")
df1.show()

+--------+---+
|    name|age|
+--------+---+
| John D.| 30|
|Alice G.| 25|
|  Bob T.| 35|
|  Eve A.| 28|
+--------+---+



In [7]:
df.show(1, vertical=True)

-RECORD 0-------------
 age        | 30      
 department | HR      
 name       | John D. 
only showing top 1 row


In [9]:
df.columns

['age', 'department', 'name']

In [20]:
print(df.age)
df.select(df.age).show()

Column<'age'>
+---+
|age|
+---+
| 30|
| 25|
| 35|
| 28|
+---+



In [18]:
df.select(['age']).show()

+---+
|age|
+---+
| 30|
| 25|
| 35|
| 28|
+---+



In [21]:
from pyspark.sql.functions import upper
df.withColumn('upper_name',upper(df['name'])).show()

+---+----------+--------+----------+
|age|department|    name|upper_name|
+---+----------+--------+----------+
| 30|        HR| John D.|   JOHN D.|
| 25|   Finance|Alice G.|  ALICE G.|
| 35|        IT|  Bob T.|    BOB T.|
| 28| Marketing|  Eve A.|    EVE A.|
+---+----------+--------+----------+



In [24]:
df=df.withColumn('upper_name',upper(df['name']))
df.show()

+---+----------+--------+----------+
|age|department|    name|upper_name|
+---+----------+--------+----------+
| 30|        HR| John D.|   JOHN D.|
| 25|   Finance|Alice G.|  ALICE G.|
| 35|        IT|  Bob T.|    BOB T.|
| 28| Marketing|  Eve A.|    EVE A.|
+---+----------+--------+----------+



In [ ]:
# to make dataframe from table in spark environment  , 'read' is a class to fetch data from any source
df = spark.read.table("table_name")

In [6]:
# to view 2 records from df
df.show(n=2)

+---+----------+--------+
|age|department|    name|
+---+----------+--------+
| 30|        HR| John D.|
| 25|   Finance|Alice G.|
+---+----------+--------+
only showing top 2 rows


In [8]:
#f we set vertical to True, the DataFrame will be displayed vertically with one line per value:
df.show(n=2,vertical=True)

-RECORD 0--------------
 age        | 30       
 department | HR       
 name       | John D.  
-RECORD 1--------------
 age        | 25       
 department | Finance  
 name       | Alice G. 
only showing top 2 rows


In [12]:
df.printSchema()

root
 |-- age: long (nullable = true)
 |-- department: string (nullable = true)
 |-- name: string (nullable = true)



In [17]:
df2=df.withColumnRenamed("department","department_name")
df2.show(n=2)

+---+---------------+--------+
|age|department_name|    name|
+---+---------------+--------+
| 30|             HR| John D.|
| 25|        Finance|Alice G.|
+---+---------------+--------+
only showing top 2 rows


In [22]:
# below both operations are same
df.filter(df['age']>30).show()
df.where(df['age']>30).show()

+---+----------+------+
|age|department|  name|
+---+----------+------+
| 35|        IT|Bob T.|
+---+----------+------+

+---+----------+------+
|age|department|  name|
+---+----------+------+
| 35|        IT|Bob T.|
+---+----------+------+



In [24]:
df.createOrReplaceTempView("employees")

### Aggregate Function

In [30]:
# aggregate function 
df.agg({"age": "max"}).show()
df.agg({"age": "sum"}).show()
df.agg({"age": "min"}).show()
df.agg({"age": "avg"}).show()

+--------+
|max(age)|
+--------+
|      35|
+--------+

+--------+
|sum(age)|
+--------+
|     118|
+--------+

+--------+
|min(age)|
+--------+
|      25|
+--------+

+--------+
|avg(age)|
+--------+
|    29.5|
+--------+



###  Grouping Data

In [28]:
df = spark.createDataFrame([
    ['red', 'banana', 1, 10], ['blue', 'banana', 2, 20], ['red', 'carrot', 3, 30],
    ['blue', 'grape', 4, 40], ['red', 'carrot', 5, 50], ['black', 'carrot', 6, 60],
    ['red', 'banana', 7, 70], ['red', 'grape', 8, 80]], schema=['color', 'fruit', 'v1', 'v2'])
df.groupby('color').avg().show()

+-----+-------+-------+
|color|avg(v1)|avg(v2)|
+-----+-------+-------+
|  red|    4.8|   48.0|
| blue|    3.0|   30.0|
|black|    6.0|   60.0|
+-----+-------+-------+



In [29]:
df=df.withColumnRenamed('fruit','Fruits')
df.show()

+-----+------+---+---+
|color|Fruits| v1| v2|
+-----+------+---+---+
|  red|banana|  1| 10|
| blue|banana|  2| 20|
|  red|carrot|  3| 30|
| blue| grape|  4| 40|
|  red|carrot|  5| 50|
|black|carrot|  6| 60|
|  red|banana|  7| 70|
|  red| grape|  8| 80|
+-----+------+---+---+



In [30]:
df.filter(df['v1']==1).show()

+-----+------+---+---+
|color|Fruits| v1| v2|
+-----+------+---+---+
|  red|banana|  1| 10|
+-----+------+---+---+



In [31]:
#Persists the DataFrame with the default storage level (MEMORY_AND_DISK_DESER).
df.cache()

DataFrame[age: bigint, department: string, name: string]

In [38]:
# return a list of Row objects, each representing a row in the DataFrame.
# collects the distributed data to the driver side as the local data in Python. 
# Note that this can throw an out-of-memory error when the dataset is too large to fit in the driver side because it collects all the data from executors to the driver side.
print(df.collect())
type(df.collect()[0])

[Row(age=30, department='HR', name='John D.'), Row(age=25, department='Finance', name='Alice G.'), Row(age=35, department='IT', name='Bob T.'), Row(age=28, department='Marketing', name='Eve A.')]


pyspark.sql.types.Row

In [13]:
df.take(1)
df.tail(1)

[Row(age=28, department='Marketing', name='Eve A.')]

In [40]:
df = spark.createDataFrame([(14, "Tom"), (23, "Alice"), (16, "Bob")], ["age", "name"])
rows = df.collect()
for row in rows:
    print(row["name"])

Tom
Alice
Bob


# Table 

A table is a persistent data structure that can be accessed across multiple Spark sessions.

In [32]:
#Note that the lifetime of this temporary table is tied to the SparkSession that was used to create this DataFrame. 
#To persist the table beyond this Spark session, you will need to save it to persistent storage.
df.createOrReplaceTempView("employees")

# Data manipulation with PySpark

In [3]:
from pyspark.sql import Row

df = spark.createDataFrame([
    Row(age=10, height=80.0, NAME="Alice"),
    Row(age=10, height=80.0, NAME="Alice"),
    Row(age=5, height=float("nan"), NAME="BOB"),
    Row(age=None, height=None, NAME="Tom"),
    Row(age=None, height=float("nan"), NAME=None),
    Row(age=9, height=78.9, NAME="josh"),
    Row(age=18, height=1802.3, NAME="bush"),
    Row(age=7, height=75.3, NAME="jerry"),
])

df.show()

+----+------+-----+
| age|height| NAME|
+----+------+-----+
|  10|  80.0|Alice|
|  10|  80.0|Alice|
|   5|   NaN|  BOB|
|NULL|  NULL|  Tom|
|NULL|   NaN| NULL|
|   9|  78.9| josh|
|  18|1802.3| bush|
|   7|  75.3|jerry|
+----+------+-----+



In [4]:
df2=df.withColumnRenamed('NAME','name')
df2.show()

+----+------+-----+
| age|height| name|
+----+------+-----+
|  10|  80.0|Alice|
|  10|  80.0|Alice|
|   5|   NaN|  BOB|
|NULL|  NULL|  Tom|
|NULL|   NaN| NULL|
|   9|  78.9| josh|
|  18|1802.3| bush|
|   7|  75.3|jerry|
+----+------+-----+



In [5]:
df.dropna(subset='NAME').show()

+----+------+-----+
| age|height| NAME|
+----+------+-----+
|  10|  80.0|Alice|
|  10|  80.0|Alice|
|   5|   NaN|  BOB|
|NULL|  NULL|  Tom|
|   9|  78.9| josh|
|  18|1802.3| bush|
|   7|  75.3|jerry|
+----+------+-----+



In [6]:
df.fillna({'age':5,'height':70}).show()

+---+------+-----+
|age|height| NAME|
+---+------+-----+
| 10|  80.0|Alice|
| 10|  80.0|Alice|
|  5|  70.0|  BOB|
|  5|  70.0|  Tom|
|  5|  70.0| NULL|
|  9|  78.9| josh|
| 18|1802.3| bush|
|  7|  75.3|jerry|
+---+------+-----+



In [7]:
# Remove outliers
df.filter(df['height'].between(65, 85)).show()

+---+------+-----+
|age|height| NAME|
+---+------+-----+
| 10|  80.0|Alice|
| 10|  80.0|Alice|
|  9|  78.9| josh|
|  7|  75.3|jerry|
+---+------+-----+



In [8]:
df.distinct().show()

+----+------+-----+
| age|height| NAME|
+----+------+-----+
|  10|  80.0|Alice|
|   5|   NaN|  BOB|
|  18|1802.3| bush|
|   9|  78.9| josh|
|   7|  75.3|jerry|
|NULL|   NaN| NULL|
|NULL|  NULL|  Tom|
+----+------+-----+



In [9]:
from pyspark.sql import functions as sf
df.withColumn('small_name',sf.lower('Name')).show()

+----+------+-----+----------+
| age|height| NAME|small_name|
+----+------+-----+----------+
|  10|  80.0|Alice|     alice|
|  10|  80.0|Alice|     alice|
|   5|   NaN|  BOB|       bob|
|NULL|  NULL|  Tom|       tom|
|NULL|   NaN| NULL|      NULL|
|   9|  78.9| josh|      josh|
|  18|1802.3| bush|      bush|
|   7|  75.3|jerry|     jerry|
+----+------+-----+----------+



In [65]:
df.select('NAME',(sf.col('age')+sf.col('height')).alias('sum age & height')).show()

+-----+----------------+
| NAME|sum age & height|
+-----+----------------+
|Alice|            90.0|
|Alice|            90.0|
|  BOB|             NaN|
|  Tom|            NULL|
| NULL|            NULL|
| josh|            87.9|
| bush|          1820.3|
|jerry|            82.3|
+-----+----------------+



In [66]:
df.filter(sf.col('age')>9).show()

+---+------+-----+
|age|height| NAME|
+---+------+-----+
| 10|  80.0|Alice|
| 10|  80.0|Alice|
| 18|1802.3| bush|
+---+------+-----+



In [67]:
from pyspark.sql import Row

df = spark.createDataFrame([
    Row(incomes=[123.0, 456.0, 789.0], NAME="Alice"),
    Row(incomes=[234.0, 567.0], NAME="BOB"),
    Row(incomes=[100.0, 200.0, 100.0], NAME="Tom"),
    Row(incomes=[79.0, 128.0], NAME="josh"),
    Row(incomes=[123.0, 145.0, 178.0], NAME="bush"),
    Row(incomes=[111.0, 187.0, 451.0, 188.0, 199.0], NAME="jerry"),
])

df.show()

+--------------------+-----+
|             incomes| NAME|
+--------------------+-----+
|[123.0, 456.0, 78...|Alice|
|      [234.0, 567.0]|  BOB|
|[100.0, 200.0, 10...|  Tom|
|       [79.0, 128.0]| josh|
|[123.0, 145.0, 17...| bush|
|[111.0, 187.0, 45...|jerry|
+--------------------+-----+



In [74]:
df_explode=df.select('name',sf.explode(sf.col('incomes')).alias('incomes'))
df_explode.show()

+-----+-------+
| name|incomes|
+-----+-------+
|Alice|  123.0|
|Alice|  456.0|
|Alice|  789.0|
|  BOB|  234.0|
|  BOB|  567.0|
|  Tom|  100.0|
|  Tom|  200.0|
|  Tom|  100.0|
| josh|   79.0|
| josh|  128.0|
| bush|  123.0|
| bush|  145.0|
| bush|  178.0|
|jerry|  111.0|
|jerry|  187.0|
|jerry|  451.0|
|jerry|  188.0|
|jerry|  199.0|
+-----+-------+



In [76]:
df_explode.groupBy('name').avg().show()

+-----+------------------+
| name|      avg(incomes)|
+-----+------------------+
|Alice|             456.0|
|  BOB|             400.5|
|  Tom|133.33333333333334|
| josh|             103.5|
|jerry|             227.2|
| bush|148.66666666666666|
+-----+------------------+



In [77]:
df_explode.groupBy('name').agg(sf.avg('incomes').alias('Averag Income')).show()

+-----+------------------+
| name|     Averag Income|
+-----+------------------+
|Alice|             456.0|
|  BOB|             400.5|
|  Tom|133.33333333333334|
| josh|             103.5|
|jerry|             227.2|
| bush|148.66666666666666|
+-----+------------------+



In [79]:
df.orderBy('name').show()

+--------------------+-----+
|             incomes| NAME|
+--------------------+-----+
|[123.0, 456.0, 78...|Alice|
|      [234.0, 567.0]|  BOB|
|[100.0, 200.0, 10...|  Tom|
|[123.0, 145.0, 17...| bush|
|[111.0, 187.0, 45...|jerry|
|       [79.0, 128.0]| josh|
+--------------------+-----+



# Joins

In [80]:
from pyspark.sql import Row

df1 = spark.createDataFrame([
    Row(age=10, height=80.0, name="alice"),
    Row(age=9, height=78.9, name="josh"),
    Row(age=18, height=82.3, name="bush"),
    Row(age=7, height=75.3, name="tom"),
])

df2 = spark.createDataFrame([
    Row(incomes=[123.0, 456.0, 789.0], name="alice"),
    Row(incomes=[234.0, 567.0], name="bob"),
    Row(incomes=[79.0, 128.0], name="josh"),
    Row(incomes=[123.0, 145.0, 178.0], name="bush"),
    Row(incomes=[111.0, 187.0, 451.0, 188.0, 199.0], name="jerry"),
])

In [81]:
df1.join(df2,on='name').show()

+-----+---+------+--------------------+
| name|age|height|             incomes|
+-----+---+------+--------------------+
|alice| 10|  80.0|[123.0, 456.0, 78...|
| bush| 18|  82.3|[123.0, 145.0, 17...|
| josh|  9|  78.9|       [79.0, 128.0]|
+-----+---+------+--------------------+



In [82]:
# There are seven join methods: - INNER - LEFT - RIGHT - FULL - CROSS - LEFTSEMI - LEFTANTI And the default one is INNER.
df1.join(df2, on="name", how="left").show()

+-----+---+------+--------------------+
| name|age|height|             incomes|
+-----+---+------+--------------------+
|alice| 10|  80.0|[123.0, 456.0, 78...|
| josh|  9|  78.9|       [79.0, 128.0]|
| bush| 18|  82.3|[123.0, 145.0, 17...|
|  tom|  7|  75.3|                NULL|
+-----+---+------+--------------------+



### UDF

In [5]:
# documentation example
from pyspark.sql.types import ArrayType, IntegerType, StringType
from pyspark.sql.functions import udf

data = [
    ("Hello World", [1, 2, 3]),
    ("PySpark is Fun", [4, 5, 6]),
    ("PySpark Rocks", [7, 8, 9])
]
df = spark.createDataFrame(data, ["text_column", "list_column"])

@udf(returnType="string")
def process_row(text: str, numbers):
    vowels_count = sum(1 for char in text if char in "aeiouAEIOU")
    doubled = [x * 2 for x in numbers]
    return f"Vowels: {vowels_count}, Doubled: {doubled}"

df.withColumn("process_row", process_row(df["text_column"], df["list_column"])).show(truncate=False)

/opt/bitnami/spark/python/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


+--------------+-----------+--------------------------------+
|text_column   |list_column|process_row                     |
+--------------+-----------+--------------------------------+
|Hello World   |[1, 2, 3]  |Vowels: 3, Doubled: [2, 4, 6]   |
|PySpark is Fun|[4, 5, 6]  |Vowels: 3, Doubled: [8, 10, 12] |
|PySpark Rocks |[7, 8, 9]  |Vowels: 2, Doubled: [14, 16, 18]|
+--------------+-----------+--------------------------------+



In [3]:
# my exmaple
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType, IntegerType

# Sample data
data = [("alice", 25), ("bob", 17), ("charlie", 30), ("diana", 15)]
df = spark.createDataFrame(data, ["name", "age"])

# Define a Python UDF
def categorize_age(age):
    if age is None:
        return "unknown"
    elif age < 18:
        return "minor"
    elif age < 65:
        return "adult"
    else:
        return "senior"

# Register the UDF
categorize_udf = udf(categorize_age, StringType())

# Use the UDF
df.withColumn("category", categorize_udf(col("age"))).show()

+-------+---+--------+
|   name|age|category|
+-------+---+--------+
|  alice| 25|   adult|
|    bob| 17|   minor|
|charlie| 30|   adult|
|  diana| 15|   minor|
+-------+---+--------+



#### UDTFs

A Python user-defined table function (UDTF) is a new kind of function that returns a table as output instead of a single scalar result value. Once registered, they can appear in the FROM clause of a SQL query. While Python UDFs in Spark are designed to each accept zero or more scalar values as input, and return a single value as output, UDTFs offer more flexibility. They can return multiple rows and columns, extending the capabilities of UDFs.